<a href="https://colab.research.google.com/github/EthanCui2008/Arxiv-Webscraper/blob/Main/Scan_and_Done_Basic_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

If this is Barr or Alisa, just sequentially click the little run buttons and something should happen


In [1]:
import google.generativeai as genai
import vertexai
from vertexai.generative_models import (
    GenerationConfig,
    GenerativeModel,
    HarmBlockThreshold,
    HarmCategory,
    Part,
    Content,
    FunctionDeclaration,
    Tool,
    ToolConfig,
)

from google.auth import credentials
from google.colab import auth

import sys
import os

import base64
import io
import requests
from PIL import Image
from io import BytesIO

In [2]:
auth.authenticate_user()

In [ ]:
LOCATION = os.environ.get("GOOGLE_CLOUD_REGION", "us-central1")
PROJECT_ID = "scan-and-done-1"
vertexai.init(project=PROJECT_ID, location=LOCATION)

In [ ]:
# Prompt: determine a set of product categories that would occur on the resale market and key features of said categories that would determine it's price, then sort those into a dictionary
product_categories = {
    "Electronics": {
        "key_features": [
            "Brand",
            "Model",
            "Storage capacity",
            "Processor",
            "RAM",
            "Condition",
            "Accessories",
            "Release date"
        ],
        "examples": ["Smartphones", "Laptops", "Tablets", "Headphones", "Cameras"]
    },
    "Clothing, Shoes & Accessories": {
        "key_features": [
            "Brand",
            "Size",
            "Condition",
            "Material",
            "Color",
            "Style"
        ],
        "examples": ["T-shirts", "Jeans", "Sneakers", "Dresses", "Watches"]
    },
    "Home & Garden": {
        "key_features": [
            "Item Name",
            "Condition",
            "Brand",
            "Material",
            "Size/Dimensions",
            "Features",
            "Age/Year of Manufacture"
        ],
        "examples": ["Furniture", "Kitchen appliances", "Decor", "Gardening tools"]
    },
    "Sporting Goods": {
        "key_features": [
            "Brand",
            "Condition",
            "Size",
            "Material",
            "Model",
            "Features"
        ],
        "examples": ["Bikes", "Golf clubs", "Camping gear", "Exercise equipment"]
    },
    "Toys & Hobbies": {
        "key_features": [
            "Brand",
            "Condition",
            "Age recommendation",
            "Features",
            "Character/Theme"
        ],
        "examples": ["Action figures", "Board games", "Building toys", "Model kits"]
    },
    "Collectibles & Antiques": {
        "key_features": [
            "Age",
            "Authenticity",
            "Condition",
            "Rarity",
            "Origin/Provenance"
        ],
        "examples": ["Stamps", "Coins", "Vintage toys", "Art"]
    }
}

In [ ]:
MODEL_ID = "gemini-1.5-pro-002"

photo_quality_checker = GenerativeModel(
    MODEL_ID,
    system_instruction=[
        "You are a helpful photo inspector",
        "Your mission is to determine if a photo of an object for sale is appropiate ",
    ],
)

product_image_describer = GenerativeModel(
    MODEL_ID,
    system_instruction=[
        "You are a helpful and very specific product descriptor",
        "Your mission is to identify classify an object with the help of a dictionary of product categories and then given the descriptors of the categoory fill in what you know and prompt the user for what you don't, for what you don't know could you provide a short list of reasonable options",
    ],
)

price_estimator = GenerativeModel(
    MODEL_ID,
    system_instruction=[
        "You are a helpful and knowledgable price estimator",
        "Your mission is given a product description try and estimate a price range for the object",
    ],
)

price_estimator = GenerativeModel(
    MODEL_ID,
    system_instruction=[
        "You are a helpful and knowledgable price estimator",
        "Your mission is given a product description try and estimate a price range for the object",
    ],
)

generation_config = GenerationConfig(
    temperature=0.5,
    top_p=1.0,
    top_k=30,
    candidate_count=1,
    max_output_tokens=8192,
)

generation_config_agent = GenerationConfig(
    temperature=0.5,
    top_p=1.0,
    top_k=30,
    candidate_count=1,
    max_output_tokens=8192,
)

safety_settings = {
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
}

def id_image(file_uri, image_type):
    prompt = "What is this image? use this dicionary" + str(product_categories)
    image_part = Part.from_uri(file_uri, mime_type="image/"+image_type)
    content = [image_part, prompt]
    response = product_image_describer.generate_content(
        content,
        generation_config=generation_config,
        safety_settings=safety_settings,
    )
    return response.text

def estimate_price(text):
    prompt = "What is the price of this item?"
    content = [text, prompt]
    print(content)
    response = price_estimator.generate_content(
        content,
        generation_config=generation_config,
        safety_settings=safety_settings,
    )


In [ ]:
response = id_image("https://i.ebayimg.com/images/g/rlYAAOSw6e9jUuX~/s-l1600.webp", "webp")
print(response)
print(estimate_price(response))

This item falls into the **Home & Garden** category, specifically a teapot, which is a type of kitchen appliance.  Here's what I can determine and what I need from you:

* **Item Name:** Teapot
* **Condition:**  (Please provide the condition. Options: New, Like New, Gently Used, Used, Fair, Poor)
* **Brand:** (Please provide the brand if known. If unknown, please say so)
* **Material:** Ceramic (appears to be glazed)
* **Size/Dimensions:** (Please provide dimensions: Height, Width (including spout and handle), Depth.  You can also provide volume/capacity.)
* **Features:** Brown with hand-painted floral design in teal, orange, red, white, and gold accents along the rim, spout, handle, and lid.
* **Age/Year of Manufacture:** (Please provide the approximate age or year of manufacture if known. If unknown, please say so)

["This item falls into the **Home & Garden** category, specifically a teapot, which is a type of kitchen appliance.  Here's what I can determine and what I need from you:

In [ ]:
MODEL_ID = "gemini-1.5-pro-002"

photo_quality_checker = GenerativeModel(
    MODEL_ID,
    system_instruction=[
        "You are a helpful photo inspector",
        "Your mission is to determine if a photo of an object for sale is appropiate ",
    ],
)

    prompt = "What is this image? use this dicionary" + str(product_categories)
    image_part = Part.from_uri(file_uri, mime_type="image/"+image_type)
    content = [image_part, prompt]
    response = product_image_describer.generate_content(
        content,
        generation_config=generation_config,
        safety_settings=safety_settings,
    )
    return response.text